# Project

<img src="images/marathon_image.png" alt="Marathon image" title="Titre de l'image" width="300" />


## Description

Our project aims to deal with marathon performance according to weather conditions.  
In order to construct the final analytical database, we collect data from three sources :
* a github repository (https://github.com/adrian3/Boston-Marathon-Data-Project) containing the csv files of the results of Boston Marathon : we collect from 2015 to 2019 ;
* wikidata to collect the data of each Boston marathon ;
* meteostat (a python library) to collect the weather data of Boston from 2015 to 2019.

At the end, the aim is to answer those three analytical queries :
* (1)
* (2)
* (3)

## Authentication

In [1]:
import requests

headers = {
    "Accept": "application/json",
    "Content-Type": "application/json"
}
body = {
  "username": "airflow",
  "password": "airflow"
}
url = f"http://airflow-apiserver:8080/auth/token"
r = requests.post(url, headers=headers, json=body)

jwt_token = r.json().get("access_token")

## Create connections between Airflow and databases

In [2]:
import requests

# MongoDB
headers = {
    "Accept": "application/json",
    "Authorization": f"Bearer {jwt_token}",
    "Content-Type": "application/json"
}
body = {
    "connection_id": "mongo_default",
    "conn_type": "mongo",
    "description": "mongo_default",
    "host": "mongo",
    "login": "admin",
    "schema": "project",
    "port": 27017,
    "password": "admin",
    "extra":  "{\"srv\": false, \"ssl\": false, \"allow_insecure\": false, \"authSource\": \"admin\"}"
}
url = f"http://airflow-apiserver:8080/api/v2/connections"
r = requests.post(url, headers=headers, json=body)

# Postgres default
body = {
    "connection_id": "postgres_default",
    "conn_type": "postgres",
    "description": "postgres_default",
    "host": "postgres",
    "login": "airflow",
    "schema": "airflow",
    "port": 5432,
    "password": "airflow",
}
url = f"http://airflow-apiserver:8080/api/v2/connections"
r = requests.post(url, headers=headers, json=body)

# Postgres Staging Zone
body = {
    "connection_id": "postgres_staging",
    "conn_type": "postgres",
    "description": "postgres_staging",
    "host": "postgres",
    "login": "airflow",
    "schema": "staging",
    "port": 5432,
    "password": "airflow",
}
url = f"http://airflow-apiserver:8080/api/v2/connections"
r = requests.post(url, headers=headers, json=body)

# Postgres Production Zone
body = {
    "connection_id": "postgres_production",
    "conn_type": "postgres",
    "description": "postgres_production",
    "host": "postgres",
    "login": "airflow",
    "schema": "production",
    "port": 5432,
    "password": "airflow",
}
url = f"http://airflow-apiserver:8080/api/v2/connections"
r = requests.post(url, headers=headers, json=body)

## Ingestion Pipelines

In [3]:
import requests
from datetime import datetime, timezone
import pytz
import time

DAG_INGESTION_IDS = ["ingest_marathons", "ingest_marathons_date", "ingest_weather"]

headers = {
    "Accept": "application/json",
    "Authorization": f"Bearer {jwt_token}",
    "Content-Type": "application/json"
}

dag_runs = {}
for dag in DAG_INGESTION_IDS :
    dt = datetime.now(timezone.utc)
    logical_date = dt.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"
    body = {
        "logical_date": logical_date,
        "conf": {}
    }
    url = f"http://airflow-apiserver:8080/api/v2/dags/{dag}/dagRuns"
    r = requests.post(url, headers=headers, json=body)
    resp_json = r.json()
    dag_runs[dag] = {
        "dag_run_id": resp_json["dag_run_id"],
        "dag_id" : resp_json["dag_id"],
        "state": ""
    }

all_done = False
while not all_done:
    print("The DAGs are running...")
    time.sleep(10)
    all_done = True
    for dag, info in dag_runs.items():
        if info["state"] not in ["success", "failed"]:
            url = f"http://airflow-apiserver:8080/api/v2/dags/{info['dag_id']}/dagRuns/{info['dag_run_id']}"
            r = requests.get(url, headers=headers)
            resp_json = r.json()
            state = resp_json["state"]
            info["state"] = state
            if state not in ["success", "failed"]:
                all_done = False    

for dag, info in dag_runs.items():
    if info["state"] == "success":
        print(f"DAG {dag} has been run successfully !")
    else:
        print(f"Error during the execution of the DAG {dag} !")

The DAGs are running...
The DAGs are running...
The DAGs are running...
The DAGs are running...
DAG ingest_marathons has been run successfully !
DAG ingest_marathons_date has been run successfully !
DAG ingest_weather has been run successfully !


## Staging pipeline

In [4]:
import requests
from datetime import datetime, timezone
import pytz
import time

headers = {
    "Accept": "application/json",
    "Authorization": f"Bearer {jwt_token}",
    "Content-Type": "application/json"
}

dag = "staging_data"
dt = datetime.now(timezone.utc)
logical_date = dt.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"
body = {
    "logical_date": logical_date,
    "conf": {}
}
url = f"http://airflow-apiserver:8080/api/v2/dags/{dag}/dagRuns"
r = requests.post(url, headers=headers, json=body)

resp_json = r.json()
dag_run_id = resp_json["dag_run_id"]
dag_id = resp_json["dag_id"]
state = ""
while state != "success" and state != "failed":
    print("The DAG is running...")
    time.sleep(10)
    url = f"http://airflow-apiserver:8080/api/v2/dags/{dag_id}/dagRuns/{dag_run_id}"
    r = requests.get(url, headers=headers)
    resp_json = r.json()
    state = resp_json["state"] 

if state == "success":
    print("The DAG has been run successfully !")
else :
    print("Error during the execution of the DAG !")

The DAG is running...
The DAG is running...
The DAG is running...
The DAG is running...
The DAG is running...
The DAG is running...
The DAG is running...
The DAG has been run successfully !


## Production pipeline

In [5]:
import requests
from datetime import datetime, timezone
import pytz
import time

headers = {
    "Accept": "application/json",
    "Authorization": f"Bearer {jwt_token}",
    "Content-Type": "application/json"
}

dag = "production_dag"
dt = datetime.now(timezone.utc)
logical_date = dt.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"
body = {
    "logical_date": logical_date,
    "conf": {}
}
url = f"http://airflow-apiserver:8080/api/v2/dags/{dag}/dagRuns"
r = requests.post(url, headers=headers, json=body)

resp_json = r.json()
dag_run_id = resp_json["dag_run_id"]
dag_id = resp_json["dag_id"]
state = ""
while state != "success" and state != "failed":
    print("The DAG is running...")
    time.sleep(10)
    url = f"http://airflow-apiserver:8080/api/v2/dags/{dag_id}/dagRuns/{dag_run_id}"
    r = requests.get(url, headers=headers)
    resp_json = r.json()
    state = resp_json["state"]
    
if state == "success":
    print("The DAG has been run successfully !")
else :
    print("Error during the execution of the DAG !")

The DAG is running...
The DAG is running...
The DAG is running...
The DAG is running...
The DAG is running...
The DAG has been run successfully !


## Execution of analytical request

In [3]:
import psycopg2

conn = psycopg2.connect(
    host="postgres",
    port=5432,
    database="airflow",
    user="airflow",
    password="airflow"
)

cur = conn.cursor()
cur.execute("SELECT version();")
print(cur.fetchone())

cur.close()
conn.close()

('PostgreSQL 16.10 (Debian 16.10-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit',)
